# 📥 Day 1: Data Ingestion & Project Setup
**Bluestock Fintech | Mutual Fund Analytics Capstone 2026**

Covers: loading all 10 raw CSVs · live NAV fetch from mfapi.in · AMFI code validation · data quality report

## 1.1 Environment Setup

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path().resolve().parent
RAW_DIR  = BASE_DIR / 'data' / 'raw'
print("Project root :", BASE_DIR)
print("Pandas:", pd.__version__, "| NumPy:", np.__version__)

## 1.2 Load All 10 Datasets

In [ ]:
DATASETS = {
    'fund_master':'01_fund_master.csv','nav_history':'02_nav_history.csv',
    'aum_by_fund_house':'03_aum_by_fund_house.csv','monthly_sip_inflows':'04_monthly_sip_inflows.csv',
    'category_inflows':'05_category_inflows.csv','industry_folio_count':'06_industry_folio_count.csv',
    'scheme_performance':'07_scheme_performance.csv','investor_transactions':'08_investor_transactions.csv',
    'portfolio_holdings':'09_portfolio_holdings.csv','benchmark_indices':'10_benchmark_indices.csv',
}
dfs, total = {}, 0
for key, fname in DATASETS.items():
    df = pd.read_csv(RAW_DIR / fname)
    dfs[key] = df; total += len(df)
    nulls = {c:v for c,v in df.isnull().sum().items() if v > 0}
    print(f"{'─'*55}")
    print(f"📁 {key.upper():<28} shape={df.shape}")
    print(f"   cols  : {list(df.columns)}")
    print(f"   nulls : {nulls if nulls else 'none'}")
    print(df.head(2).to_string(index=False))
print(f"\n{'='*55}")
print(f"Total rows across 10 datasets: {total:,}")

## 1.3 Fund Master — Categorical Summary

In [ ]:
fm = dfs['fund_master']
print("Fund Houses    :", fm['fund_house'].nunique(), "→", fm['fund_house'].unique().tolist())
print("Categories     :", fm['category'].unique().tolist())
print("Sub-categories :", fm['sub_category'].nunique(), "unique")
print("Plans          :", fm['plan'].unique().tolist())
print("Risk Grades    :", fm['risk_category'].unique().tolist())
print("\nExpense Ratio stats:")
print(fm['expense_ratio_pct'].describe().round(3).to_string())

## 1.4 Live NAV Fetch from mfapi.in

In [ ]:
import requests, json as _json, time

def fetch_nav(code, name):
    try:
        r = requests.get(f"https://api.mfapi.in/mf/{code}", timeout=10)
        r.raise_for_status()
        d = r.json()
        if d['status'] != 'SUCCESS': return None
        df = pd.DataFrame(d['data'])
        df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y')
        df['nav']  = pd.to_numeric(df['nav'], errors='coerce')
        df = df.sort_values('date').reset_index(drop=True)
        df.insert(0,'amfi_code', code)
        print(f"  ✅ {name}: {len(df):,} rows | {df['date'].min().date()} → {df['date'].max().date()}")
        return df
    except Exception as e:
        print(f"  ⚠  {name}: {e}")
        return None

SCHEMES = {'HDFC Top 100 Direct':125497,'SBI Bluechip Regular':119551,
           'ICICI Pru Bluechip':120503,'Nippon Large Cap':118632,
           'Axis Bluechip':119092,'Kotak Bluechip':120841}

print("Fetching live NAV from mfapi.in...")
live_data = {}
for name, code in SCHEMES.items():
    df = fetch_nav(code, name)
    if df is not None: live_data[name] = df
    time.sleep(0.3)

# Fallback to pre-fetched files
if not live_data:
    hdfc = pd.read_csv(RAW_DIR/'hdfc_top100_nav_live.csv', parse_dates=['date'])
    sbi  = pd.read_csv(RAW_DIR/'sbi_bluechip_nav_live.csv', parse_dates=['date'])
    live_data = {'HDFC Top 100 (pre-fetched)':hdfc, 'SBI Bluechip (pre-fetched)':sbi}
    print("  ✅ Loaded pre-fetched fallback files")

print(f"\nLive NAV datasets loaded: {len(live_data)}")

## 1.5 AMFI Code Validation

In [ ]:
nav = dfs['nav_history']
fm  = dfs['fund_master']
master_codes = set(fm['amfi_code'].astype(str))
nav_codes    = set(nav['amfi_code'].astype(str))
missing = master_codes - nav_codes
print(f"fund_master codes : {len(master_codes)}")
print(f"nav_history codes : {len(nav_codes)}")
print(f"Missing from NAV  : {sorted(missing) if missing else 'None ✅'}")
print(f"Referential integrity: {'✅ PASS' if not missing else '❌ FAIL'}")

---
## ✅ Day 1 Complete
- All 10 CSVs loaded and profiled
- Live NAV fetched from mfapi.in (fallback to pre-fetched)
- AMFI code validation: **PASS**
- No orphan codes found